# Chunking extracted documents

The next cell locates the repository, loads the extraction and chunking APIs, and extracts every supported file with usable text. The scanned notice is skipped because it has no extractable text. The example settings use local placeholders so the notebook runs without deployment credentials.

In [1]:
from pathlib import Path
from statistics import mean
import sys

from IPython.display import Markdown, display

working_dir = Path.cwd().resolve()
repo_root = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / 'app').is_dir() and (folder / 'corpus').is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError('Run this notebook from the repository or a subfolder.')
sys.path.insert(0, str(repo_root))

from app.config import Settings
from app.errors import NoExtractableTextError
from app.ingestion.chunking import chunk_document, chunk_fixed, chunk_structure
from app.ingestion.extractors import SUPPORTED_EXTENSIONS, extract_document

settings = Settings(
    _env_file=None,
    postgres_user='example',
    postgres_password='example',
    postgres_db='example',
    database_url='postgresql://localhost/example',
    groq_api_key='example',
    jwt_secret='example',
    chunk_size_words=Settings.model_fields['chunk_size_words'].default,
    chunk_overlap_words=Settings.model_fields['chunk_overlap_words'].default,
    chunk_max_words=Settings.model_fields['chunk_max_words'].default,
)
documents = {}
skipped = []
for path in sorted((repo_root / 'corpus').iterdir()):
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
        try:
            documents[path.name] = extract_document(path.read_bytes(), path.name)
        except NoExtractableTextError:
            skipped.append(path.name)

print(f'Extracted {len(documents)} documents.')
print('Skipped files without usable text:', ', '.join(skipped) or 'none')


=== Document parser messages ===



=== Document parser messages ===



=== Document parser messages ===



=== Document parser messages ===


Extracted 18 documents.
Skipped files without usable text: scanned_office_notice.pdf


## Compare chunk sizes across the corpus

The next cell runs both strategies on the same documents. It uses the project defaults of 120 words, 25 words of fixed overlap, and a 200-word structure limit. The table counts chunk content words. The embedding context header is stored separately.

In [2]:
chunks_by_strategy = {}
print('strategy  | total chunks | avg words | min | max')
print('----------+--------------+-----------+-----+----')
for strategy in ('fixed', 'structure'):
    strategy_settings = settings.model_copy(update={'chunk_strategy': strategy})
    by_filename = {
        name: chunk_document(doc, strategy_settings) for name, doc in documents.items()
    }
    chunks_by_strategy[strategy] = by_filename
    lengths = [chunk.word_count for chunks in by_filename.values() for chunk in chunks]
    print(
        f'{strategy:9} | {len(lengths):12} | {mean(lengths):9.1f}'
        f' | {min(lengths):3} | {max(lengths):3}'
    )

strategy  | total chunks | avg words | min | max
----------+--------------+-----------+-----+----
fixed     |           37 |      99.1 |  26 | 120
structure |           38 |      83.9 |   7 | 154


## Compare chunks for the expense policy

The next cell uses a 60-word target for both strategies. The fixed windows use zero overlap to make the boundary easy to see. At 60 words, a fixed boundary falls inside the Karachi/Islamabad hotel row. The structure run uses the 200-word maximum, which lets it keep the reimbursement table intact.

In [3]:
expense = documents['expense_policy.pdf']
demo_target = 60
for chunk in chunk_fixed(expense, demo_target, 0):
    print(f'Fixed chunk {chunk.position} ({chunk.word_count} words)')
    print(chunk.text, end='\n\n')

for chunk in chunk_structure(expense, demo_target, settings.chunk_max_words):
    display(Markdown(
        f'### Structure chunk {chunk.position} ({chunk.word_count} words)\n\n'
        f'{chunk.text}'
    ))

Fixed chunk 0 (60 words)
# **Tarnwick Expense and Reimbursement Policy** _Version 1.3 - Effective 1 March 2026 - Owner: Finance_ ## **1. Scope** This policy covers business expenses paid personally by employees and claimed back from Tarnwick. ## **2. Reimbursement Limits** |Expense|Limit (PKR)|Conditions| |---|---|---| |Meals in Lahore (client work)|2,500 per day|Only on client-site days| |Meals outstation|4,000 per day|Travel outside Lahore| |Hotel - Karachi /

Fixed chunk 1 (60 words)
Islamabad|15,000 per night|Pre-approval required| |Hotel - other cities|10,000 per night|Pre-approval required| |Personal car mileage|45 per km|Log start and end point| |Home internet|3,000 per month|Hybrid and remote staff only| |Client entertainment|8,000 per event|Manager approval in advance| ## **3. Travel** Air travel is economy class and is allowed only when the one-way distance is more than 400 km. Otherwise use train or

Fixed chunk 2 (60 words)
car. ## **4. Submitting Claims** Submit claims wit

### Structure chunk 0 (95 words)

# **Tarnwick Expense and Reimbursement Policy**

_Version 1.3 - Effective 1 March 2026 - Owner: Finance_

## **1. Scope**

This policy covers business expenses paid personally by employees and claimed back from Tarnwick.

## **2. Reimbursement Limits**

|Expense|Limit (PKR)|Conditions|
|---|---|---|
|Meals in Lahore (client work)|2,500 per day|Only on client-site days|
|Meals outstation|4,000 per day|Travel outside Lahore|
|Hotel - Karachi / Islamabad|15,000 per night|Pre-approval required|
|Hotel - other cities|10,000 per night|Pre-approval required|
|Personal car mileage|45 per km|Log start and end point|
|Home internet|3,000 per month|Hybrid and remote staff only|
|Client entertainment|8,000 per event|Manager approval in advance|

### Structure chunk 1 (57 words)

## **3. Travel**

Air travel is economy class and is allowed only when the one-way distance is more than 400 km. Otherwise use train or car.

## **4. Submitting Claims**

Submit claims within 30 days of the expense through ExpenseHub. Receipts are required for every item above PKR 1,000. Claims are paid with the next monthly salary.

### Structure chunk 2 (44 words)

## **5. Approval Limits**

Line managers approve claims up to PKR 50,000. Claims above PKR 50,000 need approval from the Finance Director.

Fictional document created for the RAG portfolio project. Tarnwick Consulting is an invented company.

Tarnwick Expense and Reimbursement Policy - page 1

## See the embedding input

The next cell prints the embedding input for the first company overview chunk. The filename and section title give the embedding model context for a short excerpt.

In [4]:
overview_chunk = chunks_by_strategy['structure']['company_overview.pptx'][0]
print(overview_chunk.embed_text)

Document: company_overview.pptx | Section: Tarnwick Company Overview

Tarnwick Company Overview

Internal deck - 2026


## Conclusion

Fixed chunking gives predictable window sizes and carries some context forward through overlap. It flattens line breaks and can cut through sentences or table rows. Structure chunking keeps page boundaries and can retain headings, paragraphs, and complete table rows. Its chunks vary more in size because they follow the source layout. The chunk size was reduced to about 120 words because corpus documents are typically 150 to 250 words long and larger chunks would often contain a whole document. Phase 5 will use retrieval metrics to choose the strategy.